In [36]:
print("Agent Workflow")

Agent Workflow


In [37]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [38]:
from llama_index.llms.gemini import Gemini

llm = Gemini(
    model = 'models/gemini-2.0-flash'
)

In [39]:
# Setting up tool to search the web
from tavily import AsyncTavilyClient

async def search_web(query: str) -> str:
    """
        Searches the web for a query
    """
    
    tavily_client = AsyncTavilyClient()
    return str(await tavily_client.search(query))

In [40]:
# Create Agent with Agent Workflow
from llama_index.core.agent.workflow import AgentWorkflow

workflow = AgentWorkflow.from_tools_or_functions(
    [search_web],
    llm=llm,
    system_prompt="You are a researcher who has the acces to internet you searches th web and provide detailed answers.",
)

In [41]:
# Running the Agent
response = await workflow.run(user_msg="How was the champions trophy 2025")
print(str(response))

The ICC Men's Champions Trophy 2025 is scheduled to take place from February 19 to March 9 in Pakistan and the UAE.

Here are the key details:

*   **Host:** Pakistan and UAE.
*   **Dates:** February 19 to March 9, 2025.
*   **Format:** The tournament will feature eight teams divided into two groups.
*   **Groups:**
    *   **Group A:** Bangladesh, India, New Zealand, Pakistan
    *   **Group B:** Afghanistan, Australia, England, South Africa

Some key matches include:

*   Bangladesh vs India in Dubai on February 20
*   Pakistan vs India in Dubai on February 23
*   New Zealand vs India in Dubai on March 2


In [42]:
# maintaining state
from llama_index.core.workflow import Context

ctx = Context(workflow)

response = await workflow.run(
        user_msg="Hi My name is Atharv",
        ctx=ctx,
    )

print(response)

Hello Atharv, it's nice to meet you. How can I help you with your research today?



In [43]:
response = await workflow.run(
        user_msg="what is my name",
        ctx=ctx,
    )
print(str(response))

Your name is Atharv.



Streams the output in Realtime

In [44]:
# Streaming
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream
)

handler = workflow.run("What is the weather in Pune?")

async for event in handler.stream_events():
    if isinstance(event, AgentStream):
        print(event.delta, end="", flush=True)

The weather in Pune, India on March 18, 2025, at 01:45 local time is clear with a temperature of 25.5°C (77.9°F). The wind is from the WNW at 7.6 kph, and the humidity is 27%.

The forecast for March 17, 2025, indicates a minimum temperature of 23.9°C and a maximum temperature of 35.83°C.


Maintains the State

In [53]:
# State

from llama_index.core.workflow import context

async def set_name(ctx: Context, name: str) -> str:
    state = await ctx.get('state')
    state['name'] = name
    await ctx.set("state", state)
    return f"Name of the User is {name}"

workflow = AgentWorkflow.from_tools_or_functions(
    [set_name],
    llm=llm,
    system_prompt="You are an Agent which remembers peoples name",
    initial_state={"name": "unset"},
)

ctx = Context(workflow)

response = await workflow.run("My name is KungFuPanda", ctx=ctx)
print(str(response))

state = await ctx.get("state")
print(state["name"])



Okay.

KungFuPanda


# Human in the Loop

Using workflow events, we can emit events that require a response from the user. Here, we use the built-in InputRequiredEvent and HumanResponseEvent to handle the human in the loop, but you can also define your own events.

In [55]:
from llama_index.core.workflow import (
    Context,
    InputRequiredEvent,
    HumanResponseEvent,
)

async def risky_task(ctx: Context) -> str:
    """Risky task taht requires human input"""
    ctx.write_event_to_stream(
        InputRequiredEvent(
            prefix="Are you sure you want to risk it?",
            user_name="Atharv",
        )
    )
    
    response = ctx.wait_for_event(
        HumanResponseEvent, requirements={"user_name": "Atharv"}
    )
    if response.response == "yes":
        return "Risk Hai Toh Ishq Hai"
    else:
        return "If I'm loosing now but i'm winning late thats all i want"

workflow = AgentWorkflow.from_tools_or_functions(
    [risky_task],
    llm=llm,
    system_prompt="You are trained to perforn risky tasks"
)

In [58]:
handler = workflow.run(
    user_msg="I want to take Risk",
)

async for event in handler.stream_events():
    if isinstance(event, InputRequiredEvent):
        response = input(event.prefix).strip().lower()
        handler = ctx.send_event(
            HumanResponseEvent(
                response=response,
                user_name = event.user_name,
            )
        )

response = await handler
print(str(response))

I am ready. Please proceed.

